# Demo of `LaPDXYZTransform`

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtrans
import sys

plt.rcParams["figure.figsize"] = [10.5, 0.56 * 10.5]

In [ ]:
try:
    from bapsf_motion.transform import LaPDXYZTransform
except ModuleNotFoundError:
    from pathlib import Path

    HERE = Path().cwd()
    BAPSF_MOTION = (HERE / ".." / ".." / ".." ).resolve()
    sys.path.append(str(BAPSF_MOTION))
    
    from bapsf_motion.transform import LaPDXYZTransform

General input keyword arguments to use for the demo.

These input arguments are similar to a setup on an East port of the LaPD.

In [ ]:
input_kwargs = {
    "pivot_to_center": 58.771,
    "pivot_to_xzcross": 142.4804,  # 0.81" + 54.9cm + 0.75" + 79.3cm + 1.7"
    "probe_axis_offset": 30.47,  # 0.5" + 15.1cm + 5.4cm + 8.7cm
    "table_pivot_to_zlead_screw": 12.488,  # 0.5" + 2.5cm + 4.4cm + 1.7"
    "drive_polarity": [1, -1, 1],
    "mspace_polarity": [-1, 1, -1],
}

## Transform from **Motion Space** to **Drive Space** to **Motion Space**

Let's show the transform can successfully convert from the motion space to the drive space, and back.

Instantiate the transform class.

In [ ]:
tr = LaPDXYZTransform(("x", "y", "z"), **input_kwargs)
tr.config

Construct a set of points in the motion space to convert.

`points` will be an array of points defining the boundary of an XY-plane, XZ-plane, and YZ-plane.  In that order.

In [ ]:
points = np.zeros((3*40, 3))
npoints_in_plane = 40
delta = 10

# xy-plane
points[0:10, 0] = np.linspace(-delta, delta, num=10, endpoint=False)
points[0:10, 1] = delta * np.ones(10)
points[10:20, 0] = delta * np.ones(10)
points[10:20, 1] = np.linspace(delta, -delta, num=10, endpoint=False)
points[20:30, 0] = np.linspace(delta, -delta, num=10, endpoint=False)
points[20:30, 1] = -delta * np.ones(10)
points[30:40, 0] = -delta * np.ones(10)
points[30:40, 1] = np.linspace(-delta, delta, num=10, endpoint=False)

# xz-plane
points[40:80, 0] = points[0:40, 0]
points[40:80, 2] = points[0:40, 1]

# yz-plane
points[80:, 1] = points[0:40, 0]
points[80:, 2] = points[0:40, 1]

# Define a set of "key" points, which are just the corner points
# of each plane.
key_points = np.array(
    [
        # xy-corners
        [-delta, delta, 0],
        [-delta, -delta, 0],
        [delta, -delta, 0],
        [delta, delta, 0],
        # xz-corners
        [-delta, 0, delta],
        [-delta, 0, -delta],
        [delta, 0, -delta],
        [delta, 0, delta],
        # yz-corners
        [0, -delta, delta],
        [0, -delta, -delta],
        [0, delta, -delta],
        [0, delta, delta],
    ],
)

Calculate the drive space points `dpoints` and return to motion space points `mpoints`.

In [ ]:
dpoints = tr(points, to_coords="drive")
mpoints = tr(dpoints, to_coords="motion_space")

Plot the transform

In [ ]:
figwidth, figheight = plt.rcParams["figure.figsize"]
figwidth = 1.4 * figwidth
figheight = figwidth
fig, axs = plt.subplots(
    3, 3, figsize=[figwidth, figheight], layout="constrained",
)
fig.set_constrained_layout_pads(w_pad=0.2, h_pad=0.1)

axs[0, 0].set_title("Motion Space")
axs[0, 1].set_title("Drive Space")
axs[0, 2].set_title("Drive Space")

dkp = tr(key_points, to_coords="drive")
         
for ii in range(3):
    if ii == 0:  # xy-plane
        p0 = 0
        p1 = [1, 1, 2]
        
        axs[ii, 0].set_xlabel("X")
        axs[ii, 0].set_ylabel("Y")
        
        axs[ii, 1].set_xlabel("X")
        axs[ii, 1].set_ylabel("Y")
        
        axs[ii, 2].set_xlabel("X")
        axs[ii, 2].set_ylabel("Z")
    elif ii == 1:  # xz-plane
        p0 = 0
        p1 = [2, 2, 1]

        axs[ii, 0].set_xlabel("X")
        axs[ii, 0].set_ylabel("Z")

        axs[ii, 1].set_xlabel("X")
        axs[ii, 1].set_ylabel("Z")
        
        axs[ii, 2].set_xlabel("X")
        axs[ii, 2].set_ylabel("Y")
    else:  # yz-plane
        p0 = 1
        p1 = [2, 2, 0]
        
        axs[ii, 0].set_xlabel("Y")
        axs[ii, 0].set_ylabel("Z")

        axs[ii, 1].set_xlabel("Y")
        axs[ii, 1].set_ylabel("Z")
        
        axs[ii, 2].set_xlabel("Y")
        axs[ii, 2].set_ylabel("X")
            
    i_start = ii * npoints_in_plane
    i_stop = i_start + npoints_in_plane
    
    axs[ii, 0].fill(points[i_start:i_stop, p0], points[i_start:i_stop, p1[0]])
    axs[ii, 1].fill(dpoints[i_start:i_stop, p0], dpoints[i_start:i_stop, p1[1]])
    axs[ii, 2].plot(dpoints[i_start:i_stop, p0], dpoints[i_start:i_stop, p1[2]], "-o")

    i_start = ii * 4
    i_stop = i_start +  4
    colors = ["red", "orange", "black", "purple"]

    axs[ii, 0].scatter(key_points[i_start:i_stop, p0], key_points[i_start:i_stop, p1[0]], c=colors)
    axs[ii, 1].scatter(dkp[i_start:i_stop, p0], dkp[i_start:i_stop, p1[1]], c=colors)
    axs[ii, 2].scatter(dkp[i_start:i_stop, p0], dkp[i_start:i_stop, p1[2]], c=colors, zorder=10)


# Get the bounding boxes of the axes including text decorations
r = fig.canvas.get_renderer()
get_bbox = lambda ax: ax.get_tightbbox(r).transformed(fig.transFigure.inverted())
bboxes = np.array(list(map(get_bbox, axs.flat)), mtrans.Bbox).reshape((*axs.shape, 2, 2))

# Draw vertical divider between the different space plots
x = bboxes[:, 0, 1, 0].mean() - 1.15 * (0.2 / figwidth)
line = plt.Line2D([x, x], [0,1], transform=fig.transFigure, color="gray", linestyle="--")
fig.add_artist(line);

How close are the points after the round trip conversion?  Let's plot the difference.

In [ ]:
figwidth, figheight = plt.rcParams["figure.figsize"]
figwidth = 1.4 * figwidth
fig, ax = plt.subplots(1, 1, figsize=[figwidth, figheight])

ax.plot(points[..., 0] - mpoints[..., 0], "-o", label="X")
ax.plot(points[..., 1] - mpoints[..., 1], "-o", label="Y")
ax.plot(points[..., 2] - mpoints[..., 2], "-o", label="Z")

ax.set_xlabel("Index")
ax.set_ylabel("Diff")
ax.set_title("Difference in Motion --> Drive --> Motion Conversion")
ax.legend();

Here we can see the points are virtually identical, with a difference on the order of $10^{-14}$.

In [ ]:
np.allclose(points, mpoints)

In [ ]:
np.max(np.abs(points - mpoints))

## Transform from **Drive Space** to **Motion Space** to **Drive Space**

Let's show the transform can successfully convert from the drive space to the motion space, and back.

Using the same transform and initial points in the previous section, lets construct the motion space points `mpoints` and return to drive space points `dpoints`.

In [ ]:
mpoints = tr(points, to_coords="motion_space")
dpoints = tr(mpoints, to_coords="drive")

Plot the transform.

In [ ]:
figwidth, figheight = plt.rcParams["figure.figsize"]
figwidth = 1.4 * figwidth
figheight = figwidth
fig, axs = plt.subplots(
    3, 3, figsize=[figwidth, figheight], layout="constrained",
)
fig.set_constrained_layout_pads(w_pad=0.2, h_pad=0.1)

axs[0, 0].set_title("Drive Space")
axs[0, 1].set_title("Motion Space")
axs[0, 2].set_title("Motion Space")

mkp = tr(key_points, to_coords="motion_space")
         
for ii in range(3):
    if ii == 0:  # xy-plane
        p0 = 0
        p1 = [1, 1, 2]
        
        axs[ii, 0].set_xlabel("X")
        axs[ii, 0].set_ylabel("Y")
        
        axs[ii, 1].set_xlabel("X")
        axs[ii, 1].set_ylabel("Y")
        
        axs[ii, 2].set_xlabel("X")
        axs[ii, 2].set_ylabel("Z")
    elif ii == 1:  # xz-plane
        p0 = 0
        p1 = [2, 2, 1]

        axs[ii, 0].set_xlabel("X")
        axs[ii, 0].set_ylabel("Z")

        axs[ii, 1].set_xlabel("X")
        axs[ii, 1].set_ylabel("Z")
        
        axs[ii, 2].set_xlabel("X")
        axs[ii, 2].set_ylabel("Y")
    else:  # yz-plane
        p0 = 1
        p1 = [2, 2, 0]
        
        axs[ii, 0].set_xlabel("Y")
        axs[ii, 0].set_ylabel("Z")

        axs[ii, 1].set_xlabel("Y")
        axs[ii, 1].set_ylabel("Z")
        
        axs[ii, 2].set_xlabel("Y")
        axs[ii, 2].set_ylabel("X")
            
    i_start = ii * npoints_in_plane
    i_stop = i_start + npoints_in_plane
    
    axs[ii, 0].fill(points[i_start:i_stop, p0], points[i_start:i_stop, p1[0]])
    axs[ii, 1].fill(mpoints[i_start:i_stop, p0], mpoints[i_start:i_stop, p1[1]])
    axs[ii, 2].plot(mpoints[i_start:i_stop, p0], mpoints[i_start:i_stop, p1[2]], "-o")

    i_start = ii * 4
    i_stop = i_start +  4
    colors = ["red", "orange", "black", "purple"]

    axs[ii, 0].scatter(key_points[i_start:i_stop, p0], key_points[i_start:i_stop, p1[0]], c=colors)
    axs[ii, 1].scatter(mkp[i_start:i_stop, p0], mkp[i_start:i_stop, p1[1]], c=colors)
    axs[ii, 2].scatter(mkp[i_start:i_stop, p0], mkp[i_start:i_stop, p1[2]], c=colors, zorder=10)

# Get the bounding boxes of the axes including text decorations
r = fig.canvas.get_renderer()
get_bbox = lambda ax: ax.get_tightbbox(r).transformed(fig.transFigure.inverted())
bboxes = np.array(list(map(get_bbox, axs.flat)), mtrans.Bbox).reshape((*axs.shape, 2, 2))

# Draw vertical divider between the different space plots
x = bboxes[:, 0, 1, 0].mean() - 1.15 * (0.2 / figwidth)
line = plt.Line2D([x, x], [0,1], transform=fig.transFigure, color="gray", linestyle="--")
fig.add_artist(line);

How close are the points after the round trip conversion?  Let's plot the difference.

In [ ]:
figwidth, figheight = plt.rcParams["figure.figsize"]
figwidth = 1.4 * figwidth
fig, ax = plt.subplots(1, 1, figsize=[figwidth, figheight])

ax.plot(points[..., 0] - dpoints[..., 0], "-o", label="X")
ax.plot(points[..., 1] - dpoints[..., 1], "-o", label="Y")
ax.plot(points[..., 2] - dpoints[..., 2], "-o", label="Z")

ax.set_xlabel("Index")
ax.set_ylabel("Diff")
ax.set_title("Difference in Drive --> Motion --> Drive Conversion")
ax.legend();

Here we can see the points are virtually identical, with a difference on the order of $10^{-14}$.

In [ ]:
np.allclose(points, dpoints)

In [ ]:
np.max(np.abs(points - dpoints))

## Transform Can Droop Correct

**Currently droop correction is NOT integrated but it will be.**

## Configure for West Side Deployment

**TODO: Fill this section out!!**

## The Algorithms

**TODO: Fill this section out!!**

<!--
To start we will use $(e_0, e_1)$ to represent the drive space coordinates and $(x, y)$ to represent the motion space coordinates.

<figure>
<img 
     src="LaPD6KTransform_space_relation_cartoon.png"
     style="width:100%; margin-left:auto; margin-right:auto; display:block"
     alt="top_level_cartoon">
<figcaption style="text-align:center"> Top-Level Cartoon of the Drive and Motion Space Relationship </figcaption>
</figure>

**Note:**  The motion space x-axis points towards the the LaPD -X when the probe drive is deployed on the East side of the machine.  This is why the East side operation requires `mspace_polarity = [-1, 1]`, and the West side requires `mspace_polarity = [1, 1]`.
-->

### Algorithm: Drive to Motion Space

**TODO: Fill this section out!!**

<!--
The key parameter we need to determine to convert the drive space coordinates to the motion space coordinates is the angle $\theta$, which is the angle the probe shaft makes with the horizontal.  Let's consider the following diagram...

<figure class=center>
<img 
     src="LaPD6KTransform_drive_space_overview.png"
     style="width:60%; margin-left:auto; margin-right:auto; display:block"
     alt="drive_overview"
     >
<figcaption style="text-align:center"> Drive Space Overview </figcaption>
</figure>

Here...

- $d_o$ = `probe_axis_offset` which is the perpendicular distance from the probe axis to the pinion location on the horizontal arm of the 6K probe drive.
- $R_A$ = `six_k_arm_length` which is the length of the vertical hanging arm of the 6K probe drive.
- $\beta$ is the angular drop of the pinion location from the probe drive shaft with respect to the ball valve

  $$
  tan\,\beta = \frac{d_o}{\texttt{pivot}\_\texttt{to}\_\texttt{drive}}=\frac{\texttt{probe}\_\texttt{axis}\_\texttt{offset}}{\texttt{pivot}\_\texttt{to}\_\texttt{drive}}
  $$

- $R_P$ = `pivot_to_drive_pinion` the radial distance of the probe drive pinion from the ball valve pivot
  
  $$
  R_P^2 = d_o^2 + \texttt{pivot}\_\texttt{to}\_\texttt{drive}^2
  $$
  
- The vertical pinoin location above the horizontal is given by $R_A - d_o + e_1$, assuming $e_1=0$ when the probe shaft is horizontal.
- $\gamma$ is the angle the vertical pinion makes with respect to the ball valve pivot and the horizontal

  $$
  \tan\,\gamma = \frac{R_A - d_o + e_1}{\texttt{pivot}\_\texttt{to}\_\texttt{drive}}
  $$
-->

### Algorithm: Motion to Drive Space

**TODO: Fill this section out!!**

<!--
In order to convert from the motion space to the drive space the key parameter to determine is the location of the probe drive pinion.  Again this boils down to determining the angle $\theta$, since the pinion is always a distance $R_P$ from the ball valve pivot and at an angle of $\theta - |\beta|$.

Knowing the motion space coordinates $(x, y)$ the angle $\theta$ can be written as..

$$
\tan\theta = -\frac{y}{D_C + x}
$$

Then the probe drive pinion is located at the following position with the ball valve coordinate system $(s_0, s_1)$

$$
s_0 = -R_P \cos(\theta - |\beta|)\\
s_1 = R_P \sin(\theta - |\beta|)
$$
-->